# client

> Client for interacting with the Fewsats API

In [ ]:
#| default_exp core

In [ ]:
#| export
from fastcore.utils import *
import os
import httpx
from typing import Dict, Any, List
from time import time, sleep
import json
from httpx import HTTPError

In [ ]:
#| hide 
from dotenv import load_dotenv
from fastcore.test import *

In [ ]:
#| hide
load_dotenv()

True

The `Fewsats` class handles authentication and provides the foundation for our API interactions.

In [ ]:
#| export
class Fewsats:
    "Client for interacting with the Fewsats API"
    def __init__(self,
                 api_key: str = None, # The API key for the Fewsats account
                 base_url: str = "https://hub-5n97k.ondigitalocean.app"): # The Fewsats API base URL
        self.api_key = api_key or os.environ.get("FEWSATS_API_KEY")
        if not self.api_key:
            raise ValueError("The api_key client option must be set either by passing api_key to the client or by setting the FEWSATS_API_KEY environment variable")
        self.base_url = base_url
        self._httpx_client = httpx.Client()
        self._httpx_client.headers.update({"Authorization": f"Token {self.api_key}"})


In [ ]:
k = os.getenv("FEWSATS_API_KEY")
fs = Fewsats(api_key=k)
k = os.getenv("FEWSATS_LOCAL_API_KEY")
fs = Fewsats(api_key=k, base_url="http://localhost:8000")

test_eq(fs.api_key, k)
test_eq(fs._httpx_client.headers["Authorization"], f"Token {k}")

## Methods

In [ ]:
#| export
@patch
def _request(self: Fewsats, 
             method: str, # The HTTP method to use
             path: str, # The path to request
             timeout: int = 10, # Timeout for the request in s
             **kwargs) -> Dict[str, Any]:
    "Makes an authenticated request to Fewsats API"
    url = f"{self.base_url}/{path}"
    return  self._httpx_client.request(method, url, timeout=timeout, **kwargs)

In [ ]:
r  = fs._request("GET", "v0/users/me")
test_eq(r.status_code, 200)

Let's use a helper function to process the response.

### User Info

In [ ]:
#| export

@patch
def me(self: Fewsats):
    "Retrieve the user's info."
    return self._request("GET", "v0/users/me")


In [ ]:
r = fs.me()
r.status_code, r.json()

(200,
 {'name': 'Pol',
  'last_name': 'Alvarez Vecino',
  'email': 'pol@fewsats.com',
  'billing_info': None,
  'id': 1,
  'created_at': '2024-08-20T16:13:01.255Z',
  'webhook_url': 'https://example.com/webhook'})

### Balance 

In [ ]:
#| export 

@patch
def balance(self: Fewsats):
    "Retrieve the balance of the user's wallet."
    return self._request("GET", "v0/wallets")


In [ ]:
r = fs.balance()
r.status_code, r.json()

(200, [{'id': 1, 'balance': 4457, 'currency': 'usd'}])

### Payment Methods

Retrieve the user's payment methods. Useful for checking which card will be used for purchases.

In [ ]:
#| export
@patch
def payment_methods(self: Fewsats) -> List[Dict[str, Any]]:
    "Retrieve the user's payment methods, raises an exception for error status codes."
    return self._request("GET", "v0/stripe/payment-methods")


In [ ]:
r = fs.payment_methods()
payment_methods = r.json()
r.status_code, payment_methods

(200,
 [{'id': 1,
   'last4': '4242',
   'brand': 'visa',
   'exp_month': 12,
   'exp_year': 2034,
   'is_default': False},
  {'id': 4,
   'last4': '4242',
   'brand': 'Visa',
   'exp_month': 12,
   'exp_year': 2034,
   'is_default': True}])

In [ ]:
assert isinstance(payment_methods, list)

### Preview a Purchase

Preview the resulting state of a purchase. Useful, for example, to check if a CC charge is needed or the purchase will use the balance.

In [ ]:
#| export

@patch
def _preview_payment(self: Fewsats,
                    amount: str): # The amount in USD cents
    "Simulates a purchase, raises an exception for error status codes."
    assert amount.isdigit()
    return self._request("POST", "v0/l402/preview/purchase/amount", json={"amount_usd": amount})


In [ ]:
r = fs._preview_payment(amount="300") # 3.00 USD
preview = r.json()
r.status_code = preview

### Create offers

How to use the client to generate L402 offers

In [ ]:
#| export
@patch
def create_offers(self:Fewsats,
                 offers:List[Dict[str,Any]], # List of offer objects following OfferCreateV0 schema
) -> dict:
    "Create offers for L402 payment server"
    return self._request("POST", "v0/l402/offers", json={"offers": offers})

In [ ]:
test_offers = [{
    "offer_id": "test_offer_2",
    "amount": 1,
    "currency": "USD" ,
    "description": "Test offer",
    "title": "Test Package",
    "payment_methods": ["lightning", "credit_card"]
}]

r = fs.create_offers(test_offers)
l402_offers = r.json()
r.status_code, l402_offers

(200,
 {'offers': [{'offer_id': 'test_offer_2',
    'amount': 1,
    'currency': 'USD',
    'description': 'Test offer',
    'title': 'Test Package',
    'payment_methods': ['lightning', 'credit_card'],
    'type': 'one-off'}],
  'payment_context_token': '1eed6450-2179-4df6-bffa-8ef07f4eaf17',
  'payment_request_url': 'http://localhost:8000/v0/l402/payment-request',
  'version': '0.2.2'})

### Get Payment Details

In [ ]:
#| export
@patch
def get_payment_details(self:Fewsats,
                       payment_request_url:str,
                       offer_id:str,
                       payment_method:str,
                       payment_context_token:str,
                       ) -> dict:
    data = {"offer_id": offer_id, "payment_method": payment_method, "payment_context_token": payment_context_token}
    return httpx.post(payment_request_url, json=data)


In [ ]:
r = fs.get_payment_details(l402_offers["payment_request_url"], l402_offers["offers"][0]["offer_id"], "lightning", l402_offers["payment_context_token"])
payment_details = r.json()
ln_invoice = payment_details["payment_request"]['lightning_invoice']
r.status_code, payment_details

(200,
 {'expires_at': '2025-03-10T03:57:59.546940+00:00',
  'offer_id': 'test_offer_2',
  'payment_request': {'lightning_invoice': 'lnbc120n1pnuukchpp55gu4w0jdnpk0rm0exlvn9jnvn4r20vfa2h9dhwwy3g0qxy3da2yqdq523jhxapq2pskx6mpvajscqzpgxqrzpsrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp5z2rlnpdsghhcenf04gda7rt4dz4kndmkqx3am6xgwxmu36ch2lqq9qxpqysgqw0ggfyhcv7j2e0kk3p2gux9d434ddm4swh8ldsefvm63jed3jgln27hyndssk85jzwjrxcv005fj68qcndz53np8crl7lnf5pgmha2cq0gughq'},
  'version': '0.2.2'})

### Get Payment Status


In [ ]:
#| export
@patch
def get_payment_status(self:Fewsats,
                       payment_context_token:str,
                       ) -> dict:
    return self._request("GET", f"v0/l402/payment-status?payment_context_token={payment_context_token}")

In [ ]:

r = fs.get_payment_status(l402_offers["payment_context_token"])
r.status_code, r.json()

(200,
 {'payment_context_token': '1eed6450-2179-4df6-bffa-8ef07f4eaf17',
  'status': 'pending',
  'offer_id': None,
  'paid_at': None,
  'amount': None,
  'currency': None})

In [ ]:
#| export
@patch
def set_webhook(self:Fewsats,
                       webhook_url:str,
                       ) -> dict:
    return self._request("POST", f"v0/users/webhook/set", json={"webhook_url": webhook_url})

In [ ]:

r = fs.set_webhook("https://example.com")
r, r.json()

(<Response [200 OK]>,
 {'name': 'Pol',
  'last_name': 'Alvarez Vecino',
  'email': 'pol@fewsats.com',
  'billing_info': None,
  'id': 1,
  'created_at': '2024-08-20T16:13:01.255Z',
  'webhook_url': 'https://example.com'})

### Pay Lightning Invoice

In [ ]:
#| export

@patch
def pay_lightning(self: Fewsats, 
                  invoice: str, # lightning invoice
                  amount: int, # amount in cents
                  currency: str = "USD", # currency
                  description: str = "" ): # description of the payment 
    "Pay for a lightning invoice"
    data = {
        "invoice": invoice,
        "amount": amount,
        "currency": currency,
        "description": description
    }
    return self._request("POST", "v0/l402/purchases/lightning", json=data)

In [ ]:
r = fs.pay_lightning(invoice=ln_invoice,
                     description="fewsats webhook trial", amount=1)
lightning_payment = r.json()
r.status_code, lightning_payment

(200,
 {'id': 128,
  'created_at': '2025-03-10T03:23:04.572Z',
  'status': 'success',
  'payment_request_url': '',
  'payment_context_token': '',
  'invoice': 'lnbc120n1pnuukchpp55gu4w0jdnpk0rm0exlvn9jnvn4r20vfa2h9dhwwy3g0qxy3da2yqdq523jhxapq2pskx6mpvajscqzpgxqrzpsrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp5z2rlnpdsghhcenf04gda7rt4dz4kndmkqx3am6xgwxmu36ch2lqq9qxpqysgqw0ggfyhcv7j2e0kk3p2gux9d434ddm4swh8ldsefvm63jed3jgln27hyndssk85jzwjrxcv005fj68qcndz53np8crl7lnf5pgmha2cq0gughq',
  'preimage': '4e1320983818d4dc5ef1815bddc593c27c2c2fd78c185c84ecb18ce7ab092dd4',
  'amount': 1,
  'currency': 'usd',
  'payment_method': 'lightning',
  'title': '',
  'description': 'fewsats webhook trial',
  'type': '',
  'is_test': False})

In [ ]:
fs.get_payment_status(l402_offers["payment_context_token"]).json()

{'payment_context_token': '1eed6450-2179-4df6-bffa-8ef07f4eaf17',
 'status': 'pending',
 'offer_id': None,
 'paid_at': None,
 'amount': None,
 'currency': None}

### Pay 

The pay method pays for a specific offer. The user is not required to fetch the payment details beforehand. It is asynchronous and returns the `payment_id` and `status`. Using the `payment_id` we can check the status of the payment.

In [ ]:
#| export


@patch
def pay_offer(self:Fewsats,
        offer_id : str, # the offer id to pay for
        l402_offer: Dict, # a dictionary containing L402 offers
) -> dict: # payment status response
    """Pays an offer_id from the l402_offers.

    The l402_offer parameter must be a dictionary with this structure:
    {
        'offers': [
            {
                'offer_id': 'test_offer_2',  # String identifier for the offer
                'amount': 1,                 # Numeric cost value
                'currency': 'USD',           # Currency code
                'description': 'Test offer', # Text description
                'title': 'Test Package'      # Title of the package
            }
        ],
        'payment_context_token': '60a8e027-8b8b-4ccf-b2b9-380ed0930283',  # Payment context token
        'payment_request_url': 'https://api.fewsats.com/v0/l402/payment-request',  # Payment URL
        'version': '0.2.2'  # API version
    }

    Returns payment status response"""
    data = {"offer_id": offer_id, **l402_offer}
    return self._request("POST", "v0/l402/purchases/from-offer", json=data)

In [ ]:
r = fs.pay_offer(l402_offers["offers"][0]["offer_id"], l402_offers)
payment_response = r.json() if r.is_success else r.text
r.status_code, payment_response

(200,
 {'id': 129,
  'created_at': '2025-03-10T03:23:11.321Z',
  'status': 'success',
  'payment_method': 'lightning'})

After the stripe payment settles, the status will be updated to `success`.

### Payment Info

We can check the status of a payment as follows:

In [ ]:
#| export
@patch
def payment_info(self:Fewsats,
                  pid:str): # purchase id
    "Retrieve the details of a payment."
    return self._request("GET", f"v0/l402/outgoing-payments/{pid}")

In [ ]:
r = fs.payment_info(payment_response['id'])
r.status_code, r.json()

(200,
 {'id': 129,
  'created_at': '2025-03-10T03:23:11.321Z',
  'status': 'success',
  'payment_request_url': 'http://localhost:8000/v0/l402/payment-request',
  'payment_context_token': '1eed6450-2179-4df6-bffa-8ef07f4eaf17',
  'invoice': 'lnbc120n1pnuukc7pp558pekz2h609sxmrw5flz9uw09mzudzuh6r0g7qnrj8cpzg3jnp3sdq523jhxapq2pskx6mpvajscqzpgxqrzpjrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp55te447m4jxxh22y2l7wn4gxy9zrnwe7ykjyjh2wngduftgq5cy9s9qxpqysgqwdd3t263zcdqsnzvd9frlmc99xlxayx7xxgunx7suygs0d85dnpz3kt800nzu9vay9gq4c8kh4jraz6m7h8xpv7p894wxsn7kgj6reqqq5ca7k',
  'preimage': 'e6508fc118525c65f224e4858b7935fa1b29d2a0fbb62196c4f9fa9c16579de0',
  'amount': 1,
  'currency': 'usd',
  'payment_method': 'lightning',
  'title': 'Test Package',
  'description': 'Test offer',
  'type': 'one-off',
  'is_test': False})

## As tools

In [ ]:
#| export

@patch
def as_tools(self:Fewsats):
    "Return list of available tools for AI agents"
    return [
        self.me,
        self.balance,
        self.payment_methods,
        self.pay_lightning,
        self.payment_info,
    ]

In [ ]:
fs.as_tools()

[<bound method Fewsats.me of <__main__.Fewsats object>>,
 <bound method Fewsats.balance of <__main__.Fewsats object>>,
 <bound method Fewsats.payment_methods of <__main__.Fewsats object>>,
 <bound method Fewsats.pay_lightning of <__main__.Fewsats object>>,
 <bound method Fewsats.payment_info of <__main__.Fewsats object>>]

Both the preview and purchase methods automatically use the default payment method if a charge is needed. This client provides a straightforward way to interact with the Fewsats API, making it easy for developers to integrate Fewsats functionality into their applications.

## Agent Demo

We will use [Claudette](https://claudette.answer.ai/) to demonstrate how to pay for content using the Fewsats API.

In [ ]:
from claudette import Chat, models

In [ ]:
model = models[1]
model

'claude-3-5-sonnet-20240620'

In [ ]:
fs.balance()

<Response [200 OK]>

In [ ]:
chat = Chat(model, sp='You are a helpful assistant that can pay offers.', tools=fs.as_tools())
pr = f"Could you pay the cheapest offer using lightning {l402_offers}?"
r = chat.toolloop(pr, trace_func=print)
r

Message(id='msg_01W1TGPkmXMYjHNASw1j31AT', content=[TextBlock(text="Certainly! I'll analyze the offer information you've provided and proceed with paying the cheapest offer using Lightning Network.\n\nFrom the information you've given, there's only one offer available:\n\nOffer ID: test_offer_2\nAmount: 1 USD\nDescription: Test offer\nTitle: Test Package\nPayment Methods: Lightning and Credit Card\nType: One-off\n\nSince this is the only offer and you've requested to pay the cheapest one, we'll proceed with this offer using the Lightning Network payment method.\n\nTo pay this offer, we need to use the `pay_lightning` function. However, before we can do that, we need to obtain the Lightning invoice. The payment_request_url provided in your information suggests where we can get this invoice, but we don't have a direct function to fetch it.\n\nGiven the limitations, let's proceed with the information we have and use the `pay_lightning` function. We'll need to assume that the Lightning inv

It appears that we were able to check your balance successfully. However, I don't have the actual balance information to share with you. 

Given the error we encountered and the lack of a proper Lightning invoice, I suggest the following steps:

1. Confirm that you have sufficient balance in your wallet to cover the 1 USD payment.
2. Obtain the actual Lightning invoice from the payment_request_url you provided: http://localhost:8000/v0/l402/payment-request
3. Once you have the correct Lightning invoice, please provide it to me, and I'll attempt the payment again.

Is there anything else you'd like me to do or check regarding this payment?

<details>

- id: `msg_01NihZpPtjb5G2V4XDcUP49d`
- content: `[{'text': "It appears that we were able to check your balance successfully. However, I don't have the actual balance information to share with you. \n\nGiven the error we encountered and the lack of a proper Lightning invoice, I suggest the following steps:\n\n1. Confirm that you have sufficient balance in your wallet to cover the 1 USD payment.\n2. Obtain the actual Lightning invoice from the payment_request_url you provided: http://localhost:8000/v0/l402/payment-request\n3. Once you have the correct Lightning invoice, please provide it to me, and I'll attempt the payment again.\n\nIs there anything else you'd like me to do or check regarding this payment?", 'type': 'text'}]`
- model: `claude-3-5-sonnet-20240620`
- role: `assistant`
- stop_reason: `end_turn`
- stop_sequence: `None`
- type: `message`
- usage: `{'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 1469, 'output_tokens': 152}`

</details>

We can see in the chat history to see that the agent correctlye filled the required information for the payment.

The payment balance has also decreased as expected.

In [ ]:
fs.balance(), chat.h

(<Response [200 OK]>,
 [{'role': 'user',
   'content': [{'type': 'text',
     'text': "Could you pay the cheapest offer using lightning {'offers': [{'offer_id': 'test_offer_2', 'amount': 1, 'currency': 'USD', 'description': 'Test offer', 'title': 'Test Package', 'payment_methods': ['lightning', 'credit_card'], 'type': 'one-off'}], 'payment_context_token': '1eed6450-2179-4df6-bffa-8ef07f4eaf17', 'payment_request_url': 'http://localhost:8000/v0/l402/payment-request', 'version': '0.2.2'}?"}]},
  {'role': 'assistant',
   'content': [TextBlock(text="Certainly! I'll analyze the offer information you've provided and proceed with paying the cheapest offer using Lightning Network.\n\nFrom the information you've given, there's only one offer available:\n\nOffer ID: test_offer_2\nAmount: 1 USD\nDescription: Test offer\nTitle: Test Package\nPayment Methods: Lightning and Credit Card\nType: One-off\n\nSince this is the only offer and you've requested to pay the cheapest one, we'll proceed with this

In [ ]:
#|hide
from nbdev.doclinks import nbdev_export
nbdev_export()